In [ ]:
import sys
from pathlib import Path
import duckdb

sys.path.insert(0, str(Path.cwd().parent))

In [1]:
from paths import DEF_GLOB

In [115]:
con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")

**56,882,528** records across all instrument classes (C, P, T, M).
**125,313** unique contracts across all instrument classes (C, P, T, M).

In [116]:
con.sql(f"""
    SELECT
        COUNT(*)                        AS n_rows,
        COUNT(DISTINCT instrument_id)   AS n_contracts
    FROM read_parquet('{DEF_GLOB}')
""").df()

,n_rows,n_contracts
0,56882528,125313


Date range: **2019-03-03 to 2026-06-28**
Definition covers full 7-year history.

In [117]:
con.sql(f"""
    SELECT 
        MIN(ts_event) as earliest,
        MAX(ts_event) as latest
    FROM read_parquet('{DEF_GLOB}')
""").df()

,earliest,latest
0,2019-03-03 00:00:00+00:00,2026-06-28 00:00:00+00:00


**Instrument Classes**
Four types:
1. C (Call)
2. P (Put)
3. T (Futures Spread)
4. M (Multi-leg). 

_Only C and P are relevant for option pricing._

In [118]:
con.sql(f"""
    SELECT instrument_class, COUNT(*) as count
    FROM read_parquet('{DEF_GLOB}')
    GROUP BY instrument_class
""").df()

,instrument_class,count
0,C,7502927
1,P,7502927
2,M,411532
3,T,41465142


**T Class — Futures Spreads**

T = calendar spread between two futures months. 
Symbol contains a dot between two contract codes e.g. `FQV0020.Z0020`. 

**No strike price** → not useful for option pricing.

In [119]:
con.sql(f"""
    SELECT 
        instrument_class, 
        raw_symbol, 
        MIN(ts_event) as first_seen, 
        strike_price
    FROM read_parquet('{DEF_GLOB}')
    WHERE instrument_class IN ('T')
    GROUP BY instrument_class, raw_symbol, strike_price
    LIMIT 5
""").df()

,instrument_class,raw_symbol,first_seen,strike_price
0,T,TFO FSJ0019.U0019_OSCE0000023502032,2019-03-03 00:00:00+00:00,NaN
1,T,TFO FQJ0019.M0019_OQPE0000012002032,2019-03-03 00:00:00+00:00,NaN
2,T,TFO FQJ0019.M0019_OQCE0000012502032,2019-03-03 00:00:00+00:00,NaN
3,T,TFO FQJ0019.M0019_OQCE0000007002032,2019-03-06 00:00:00+00:00,NaN
4,T,TFO FSJ0019.U0019_OSCE0000014502032,2019-03-03 00:00:00+00:00,NaN


**M Class — Multi-leg Strategies**

M = combination strategies (straddles, strangles etc). 
Numeric code symbol with no strike or expiry encoding. 

**No strike price** → not useful for option pricing.

In [120]:
con.sql(f"""
    SELECT 
        instrument_class, 
        raw_symbol, 
        MIN(ts_event) as first_seen,
        strike_price
    FROM read_parquet('{DEF_GLOB}')
    WHERE instrument_class IN ('M')
    GROUP BY instrument_class, raw_symbol, strike_price
    LIMIT 5
""").df()

,instrument_class,raw_symbol,first_seen,strike_price
0,M,TFO 56 30707416,2024-01-22 00:00:00+00:00,NaN
1,M,TFO 73 30707617,2024-01-22 00:00:00+00:00,NaN
2,M,TFO 56 30707774,2024-01-22 00:00:00+00:00,NaN
3,M,TFO 58 30707834,2024-01-22 00:00:00+00:00,NaN
4,M,TFO 56 30707859,2024-01-22 00:00:00+00:00,NaN


**15,005,854** records across instrument classes (C, P). 

**35,952** unique contracts across instrument classes (C, P).

In [121]:
con.sql(f"""
    SELECT
        COUNT(*)                      AS n_rows,
        COUNT(DISTINCT instrument_id) AS n_contracts
    FROM read_parquet('{DEF_GLOB}')
    WHERE instrument_class IN ('C', 'P')
""").df()

,n_rows,n_contracts
0,15005854,35952


**Calls and Puts - Working Dataset**

Filter to instrument_class IN ('C', 'P'). 
All **15,005,854** rows of actual options. 
Strikes and expiries are pre-parsed by Databento.

In [122]:
con.sql(f"""
    SELECT 
        raw_symbol,
        strike_price,
        expiration,
        instrument_class
    FROM read_parquet('{DEF_GLOB}')
    WHERE instrument_class IN ('C', 'P')
    LIMIT 5
""").df()

,raw_symbol,strike_price,expiration,instrument_class
0,TFO FMJ0019_OMCE0000013002032719,13.0,2019-03-27 00:00:00+00:00,C
1,TFO FMJ0019_OMPE0000013002032719,13.0,2019-03-27 00:00:00+00:00,P
2,TFO FMJ0019_OMCE0000013502032719,13.5,2019-03-27 00:00:00+00:00,C
3,TFO FMJ0019_OMPE0000013502032719,13.5,2019-03-27 00:00:00+00:00,P
4,TFO FMJ0019_OMCE0000014002032719,14.0,2019-03-27 00:00:00+00:00,C


**100% FM prefix**

All **15,005,854** options are on front-month TTF futures.

In [123]:
con.sql(f"""
    SELECT 
        SUBSTRING(raw_symbol, 5, 2) as underlying_prefix,
        COUNT(*) as count
    FROM read_parquet('{DEF_GLOB}')
    WHERE instrument_class IN ('C', 'P')
    GROUP BY underlying_prefix
    ORDER BY count DESC
""").df()

,underlying_prefix,count
0,FM,15005854


**Exercise Style Verification**
1. **OCEXPS** = European Call 
2. **OPEXPS** = European Put.

In [124]:
con.sql(f"""
    SELECT 
        cfi,
        instrument_class,
        COUNT(*) as count
    FROM read_parquet('{DEF_GLOB}')
    WHERE instrument_class IN ('C', 'P')
    GROUP BY cfi, instrument_class
    ORDER BY count DESC
""").df()

,cfi,instrument_class,count
0,OCEXPS,C,7502927
1,OPEXPS,P,7502927


**Strike prices**
- Range: €0.50 to €1000/MWh
- 510 unique strikes
- Average €75.71 reflects full history including
    - pre-crisis (€10-30)
    - crisis (€50-340) 
    - post-crisis (€20-50)

Critical for payoff formula: max(P_T - K, 0).

In [125]:
con.sql(f"""
    SELECT 
        MIN(strike_price) as min_strike,
        MAX(strike_price) as max_strike,
        COUNT(DISTINCT strike_price) as unique_strikes,
        AVG(strike_price) as avg_strike
    FROM read_parquet('{DEF_GLOB}')
    WHERE instrument_class IN ('C', 'P')
""").df()

,min_strike,max_strike,unique_strikes,avg_strike
0,0.5,1000.0,510,75.713064


**Expiry date**
- 142 unique expiry dates (one per month, March 2019 to November 2030). 

Critical for time-to-expiry calculation T = (expiration - today) / 365.

In [126]:
con.sql(f"""
    SELECT 
        COUNT(DISTINCT expiration) as unique_expiries,
        MIN(expiration) as earliest_expiry,
        MAX(expiration) as latest_expiry
    FROM read_parquet('{DEF_GLOB}')
    WHERE instrument_class IN ('C', 'P')
""").df()

,unique_expiries,earliest_expiry,latest_expiry
0,142,2019-03-27 00:00:00+00:00,2030-11-26 00:00:00+00:00
